# Watershed e SLIC — Gustavo

Roteiro inicial: algoritmos e resultados ainda não implementados.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src").is_dir() and (p / "requirements.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abra o Jupyter dentro da pasta do projeto.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "originais"
RESULTADOS = ROOT / "resultados"
print("Projeto:", ROOT)


## 1. Implementar e validar Watershed

**Ideia:** a imagem vira um relevo. A altura de cada pixel é a magnitude do gradiente de Sobel, alta nas bordas e baixa dentro das regiões. A água sobe a partir de **marcadores** (fontes) e cada bacia cresce até encontrar outra; o encontro das bacias é o contorno.

**Etapas** (`src/segmentacao.py`):
1. `para_cinza`: o relevo precisa de um valor por pixel.
2. `gradiente_sobel`: altura do relevo, sem normalizar, para manter a mesma escala nas imagens original, ruidosa e filtrada.
3. `marcadores`: regiões planas (gradiente ≤ limiar) com uma área mínima. Sem esse filtro, cada mínimo local vira uma região (supersegmentação).
4. `watershed`: inundação por fila de prioridade, adaptada do exemplo da aula. O pixel mais baixo sai primeiro e passa o rótulo aos vizinhos.
5. `contornos`: pixels onde o rótulo muda.

`segmentar_watershed` junta as etapas. Os parâmetros são **relativos à própria imagem**, porque o enunciado avisa que o professor pode testar qualquer imagem na apresentação:
- `PERCENTIL = 75`: o limiar é o percentil 75 do gradiente, ou seja, os 75% de pixels mais planos podem virar marcador. Um limiar fixo (testamos 0,1) funcionou no carro, mas com ruído σ = 0,05 deixou só 2 marcadores e, em outras imagens do material da aula, nenhum.
- `FRAC_AREA = 0.001`: a região plana precisa ter pelo menos 0,1% dos pixels da imagem (168 px no carro), seja a imagem pequena ou grande.

Validação em entradas pequenas de resposta conhecida: `tests/test_segmentacao.py`.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "unittest", "tests.test_segmentacao"], capture_output=True, text=True, cwd=ROOT)
print(r.stderr)

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from src.segmentacao import segmentar_watershed, contornos

CAMINHO_IMAGEM = DATA / "2009_003974.jpg"   # troque para testar outra imagem
PERCENTIL = 75      # limiar = percentil do gradiente
FRAC_AREA = 0.001   # área mínima do marcador = 0,1% dos pixels

with Image.open(CAMINHO_IMAGEM) as entrada:
    original_rgb = np.asarray(entrada.convert("RGB"), dtype=np.float64) / 255.0

def sobrepor(rgb, rotulos):
    saida = rgb.copy()
    saida[contornos(rotulos)] = [1, 0, 0]
    return saida

grad, marcas, rot_ws = segmentar_watershed(original_rgb, PERCENTIL, FRAC_AREA)
print("Marcadores:", marcas.max(), "| Regiões:", len(np.unique(rot_ws)))

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].imshow(original_rgb); ax[0].set_title("Original")
ax[1].imshow(grad, cmap="gray"); ax[1].set_title("Relevo: gradiente de Sobel")
ax[2].imshow(np.ma.masked_equal(marcas, 0), cmap="tab20"); ax[2].set_title(f"Marcadores ({marcas.max()})")
ax[3].imshow(sobrepor(original_rgb, rot_ws)); ax[3].set_title(f"Watershed: {len(np.unique(rot_ws))} regiões")
for a in ax: a.axis("off")
plt.tight_layout()
(RESULTADOS / "segmentacao").mkdir(parents=True, exist_ok=True)
plt.savefig(RESULTADOS / "segmentacao" / "watershed_original.png", dpi=120, bbox_inches="tight")
plt.show()

### Por que os marcadores importam

Sem área mínima (1 pixel basta), toda pequena região plana vira uma fonte de água, e o resultado é a **supersegmentação**, o principal defeito do Watershed. A figura compara as duas configurações com o mesmo percentil.

In [ ]:
_, marcas_1, rot_1 = segmentar_watershed(original_rgb, PERCENTIL, frac_area=0)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(sobrepor(original_rgb, rot_1)); ax[0].set_title(f"Sem área mínima: {marcas_1.max()} regiões")
ax[1].imshow(sobrepor(original_rgb, rot_ws)); ax[1].set_title(f"Área mínima {FRAC_AREA:.1%}: {marcas.max()} regiões")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig(RESULTADOS / "segmentacao" / "watershed_supersegmentacao.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Implementar e validar SLIC

**Ideia:** um k-means local. Os centros começam numa grade de passo $S = \sqrt{N/K}$ e cada pixel se junta ao centro mais próximo **em cor e em posição**, procurando só numa janela $2S \times 2S$ em volta de cada centro:

$$D = \sqrt{d_c^2 + \left(\frac{m}{S}\right)^2 d_s^2}$$

Depois cada centro vai para a média de posição e cor dos seus pixels, e o processo se repete (10 iterações, como na aula).

- `N_SUPERPIXELS = 200`: quantidade aproximada de regiões (K).
- `M = 20`: compacidade. Com $m$ grande a posição domina e os superpixels viram blocos regulares; com $m$ pequeno a cor domina e os contornos seguem as bordas, mas ficam irregulares. As cores ficam na escala 0–255, como no exemplo da aula; 20 é o meio-termo visual entre 5 e 40 (figura abaixo).

**Diferenças em relação à versão da aula:**
1. Cada janela é calculada de uma vez com NumPy, e não pixel a pixel. O algoritmo é o mesmo, mas leva menos de 1 s no carro.
2. `unir_fragmentos`: passo final do SLIC original (Achanta et al., 2012), que não está no exemplo da aula. O k-means não obriga um rótulo a ser uma peça só: no carro, 216 rótulos viravam até 3.874 pedaços desconexos, o que falsearia o número de regiões. Fragmentos com menos de $S^2/4$ pixels são unidos ao vizinho.

**Watershed × SLIC, na ideia:** o Watershed decide o número de regiões pelo conteúdo da imagem (quantos vales planos existem); o SLIC recebe o número de regiões como parâmetro e divide a imagem em pedaços de tamanho parecido. Por isso o SLIC é uma *pré-segmentação* (superpixels), não uma separação de objetos.

In [ ]:
from src.segmentacao import slic

N_SUPERPIXELS = 200
M = 20

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
for a, m in zip(ax, (5, M, 40)):
    rot = slic(original_rgb, N_SUPERPIXELS, m)
    if m == M:
        rot_slic = rot
    a.imshow(sobrepor(original_rgb, rot)); a.set_title(f"SLIC m = {m}: {rot.max()} regiões"); a.axis("off")
plt.tight_layout()
plt.savefig(RESULTADOS / "segmentacao" / "slic_compacidade.png", dpi=120, bbox_inches="tight")
plt.show()
print("SLIC escolhido (m = %d): %d regiões" % (M, rot_slic.max()))

## 3. Executar nas imagens original, ruidosa e filtrada

Os dois métodos rodam com **os mesmos parâmetros** nas três imagens (`PERCENTIL`, `FRAC_AREA`, `N_SUPERPIXELS`, `M` das seções 1 e 2). Assim, a diferença entre os resultados vem só da imagem.

As três imagens vêm da filtragem da Born, em `resultados/frequencia/entradas_segmentacao.npz`: ruído gaussiano σ = 0,08 com seed 42 e passa-baixa gaussiano com corte 0,25 ciclos/pixel, escolhido pelo maior PSNR (parâmetros em `parametros.json`). Usamos os arrays float64 do `.npz`, e não os PNGs, que são arredondados para 8 bits. Não geramos ruído novo, para que segmentação e PSNR falem das mesmas imagens.

In [ ]:
import json

# Para outra imagem: python scripts/experimento_frequencia.py --imagem CAMINHO --destino resultados/teste
ENTRADAS = RESULTADOS / "frequencia"
dados = np.load(ENTRADAS / "entradas_segmentacao.npz")
parametros = json.loads((ENTRADAS / "parametros.json").read_text(encoding="utf-8"))
assert dados["original_rgb"].shape == original_rgb.shape and np.allclose(dados["original_rgb"], original_rgb), \
    "As entradas da Born são de outra imagem: gere-as para CAMINHO_IMAGEM com o comando acima."
ruidosa_rgb, filtrada_rgb = dados["ruidosa_rgb"], dados["filtrada_rgb"]
e = parametros["escolha"]
print(f"Ruído σ = {parametros['sigma']}, seed {parametros['seed']} | filtro {e['filtro']}, corte {e['corte']} ciclos/pixel")
print(f"PSNR ruidosa {parametros['psnr_ruidosa_db']:.2f} dB -> filtrada {e['psnr_db']:.2f} dB")

In [ ]:
import time

IMAGENS = {"Original": original_rgb, "Ruidosa": ruidosa_rgb, "Filtrada": filtrada_rgb}
METODOS = ("Watershed", "SLIC")
segs = {}
for nome, img in IMAGENS.items():
    t0 = time.perf_counter()
    ws = segmentar_watershed(img, PERCENTIL, FRAC_AREA)[2]
    t1 = time.perf_counter()
    sl = slic(img, N_SUPERPIXELS, M)
    t2 = time.perf_counter()
    segs[nome] = {"Watershed": ws, "SLIC": sl}
    print(f"{nome:9s} | Watershed {ws.max():4d} regiões ({t1 - t0:.1f} s) | SLIC {sl.max():4d} regiões ({t2 - t1:.1f} s)")

fig, ax = plt.subplots(2, 3, figsize=(18, 8))
for j, nome in enumerate(IMAGENS):
    for i, metodo in enumerate(METODOS):
        rot = segs[nome][metodo]
        ax[i, j].imshow(sobrepor(IMAGENS[nome], rot))
        ax[i, j].set_title(f"{metodo} | {nome}: {rot.max()} regiões")
        ax[i, j].axis("off")
plt.tight_layout()
plt.savefig(RESULTADOS / "segmentacao" / "comparacao_regioes.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Comparar regiões e sobrepor contornos

**Não existe gabarito** para esta imagem (`segmented=0` no XML do VOC, ver `data/README.md`). Então a referência de cada método é **a segmentação que ele mesmo faz na original limpa**. A pergunta respondida é: *quanto o ruído desloca os contornos, e quanto o filtro recupera?* Isso mede estabilidade, não acerto em relação a uma segmentação humana.

Métricas (`comparar_contornos`, mesma ideia do benchmark BSDS, Martin et al., 2004), com tolerância de 2 px, porque um contorno deslocado em 1 px ainda é o mesmo contorno:
- **Número de regiões**, pedido pelo enunciado.
- **Precisão:** fração do contorno obtido que também existe na referência. Baixa = **bordas falsas**, criadas pelo ruído.
- **Revocação:** fração do contorno da referência que foi reencontrado. Baixa = **bordas perdidas**.
- **F1:** média harmônica das duas, para resumir num número só.

Na figura de zoom, **amarelo** = contorno que coincide com a referência, **vermelho** = borda falsa ou deslocada (derruba a precisão), **verde** = borda da referência que sumiu (derruba a revocação).

In [ ]:
import csv
from src.segmentacao import comparar_contornos

TOLERANCIA = 2  # px

linhas = []
print(f"{'Método':10s} {'Imagem':9s} {'Regiões':>7s} {'Precisão':>9s} {'Revocação':>10s} {'F1':>5s}")
for metodo in METODOS:
    referencia = segs["Original"][metodo]
    for nome in IMAGENS:
        rot = segs[nome][metodo]
        p, r, f = comparar_contornos(rot, referencia, TOLERANCIA)
        linhas.append((metodo, nome, int(rot.max()), round(p, 3), round(r, 3), round(f, 3)))
        print(f"{metodo:10s} {nome:9s} {rot.max():7d} {p:9.2f} {r:10.2f} {f:5.2f}")

# Tabela para a Born juntar com PSNR/SSIM no notebook final.
with open(RESULTADOS / "segmentacao" / "tabela_segmentacao.csv", "w", newline="", encoding="utf-8") as arq:
    escritor = csv.writer(arq)
    escritor.writerow(["metodo", "imagem", "regioes", "precisao", "revocacao", "f1"])
    escritor.writerows(linhas)

In [ ]:
def sobrepor_referencia(rgb, rotulos, referencia):
    saida = rgb.copy()
    c, ref = contornos(rotulos), contornos(referencia)
    saida[ref & ~c] = [0, 1, 0]  # verde: borda perdida
    saida[c & ~ref] = [1, 0, 0]  # vermelho: borda falsa ou deslocada
    saida[c & ref] = [1, 1, 0]   # amarelo: coincide com a referência
    return saida

ZOOM = (slice(110, 310), slice(260, 500))  # frente do carro: roda, capô e para-brisa
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
for i, metodo in enumerate(METODOS):
    for j, nome in enumerate(("Ruidosa", "Filtrada")):
        _, _, f = comparar_contornos(segs[nome][metodo], segs["Original"][metodo], TOLERANCIA)
        quadro = sobrepor_referencia(original_rgb, segs[nome][metodo], segs["Original"][metodo])
        ax[i, j].imshow(quadro[ZOOM])
        ax[i, j].set_title(f"{metodo} | {nome} × Original (F1 = {f:.2f})")
        ax[i, j].axis("off")
plt.tight_layout()
plt.savefig(RESULTADOS / "segmentacao" / "comparacao_contornos_zoom.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Registrar metodologia, parâmetros e limitações

**Metodologia da comparação visual:**
1. Mesma imagem de entrada (`2009_003974.jpg`) e mesmos parâmetros nas três versões: original, ruidosa e filtrada.
2. Contornos em vermelho sobre a imagem (seção 3), e o número de regiões no título.
3. Referência de cada método = sua segmentação da original limpa. No zoom, a cor diz se a borda coincide, é falsa ou foi perdida (seção 4).
4. Números na tabela (`resultados/segmentacao/tabela_segmentacao.csv`), juntados com PSNR/SSIM no notebook final.

**Parâmetros:**

| Método | Parâmetro | Valor | Por quê |
|---|---|---|---|
| Watershed | limiar dos marcadores | percentil 75 do gradiente | relativo à imagem; um valor fixo falhou com ruído e em outras imagens |
| Watershed | área mínima do marcador | 0,1% dos pixels | evita a supersegmentação (791 regiões sem ela) |
| SLIC | superpixels (K) | 200 | detalhe suficiente sem ficar lento |
| SLIC | compacidade (m) | 20 | meio-termo entre seguir bordas (m = 5) e virar grade (m = 40) |
| SLIC | iterações | 10 | como na aula |
| Métrica | tolerância de contorno | 2 px | um contorno deslocado em 1–2 px ainda é o mesmo |

**O que a tabela mostra** (entradas da Born: ruído σ = 0,08; filtro gaussiano com corte 0,25 ciclos/pixel):
- **Watershed perde bordas.** As regiões caem de 20 para 9 e a revocação para 0,55, mas a precisão continua alta (0,89). O ruído quebra as regiões planas em pedaços menores que a área mínima, sobram menos marcadores e bacias vizinhas se fundem. As bordas que sobram são as fortes.
- **SLIC inventa bordas.** A precisão cai para 0,47: os contornos ficam serrilhados e caem fora da referência. Com esse ruído forte, o número de regiões também cai (203 → 148): 72 dos 216 superpixels ficam pulverizados em pedaços menores que $S^2/4$ e são absorvidos pelos vizinhos (na original, 21). Com ruído moderado (σ = 0,05, testado durante o desenvolvimento) o SLIC mantinha cerca de 200 regiões; ou seja, contar regiões subestima o efeito do ruído no SLIC.
- **O filtro recupera o SLIC, quase nada do Watershed.** SLIC: F1 0,56 → 0,74 e 206 regiões. Watershed: F1 0,68 → 0,69 e 11 regiões. O corte escolhido pelo maior PSNR é suave e deixa ruído suficiente para continuar quebrando as regiões planas que viram marcadores.
- **PSNR × segmentação:** o filtro ganha 3,5 dB de PSNR (22,6 → 26,1 dB), mas o F1 do Watershed sobe só 0,01. PSNR melhor não garante segmentação melhor; por isso medimos os contornos.

**Limitações:**
- Sem gabarito humano, a métrica mede **estabilidade frente ao ruído**, e não acerto. Uma borda que o método erra na original também é "referência".
- No fundo liso, as fronteiras do SLIC são arbitrárias (seguem a grade inicial); qualquer perturbação as desloca, o que penaliza a precisão do SLIC mesmo onde não há objeto.
- Uma imagem só e uma única realização do ruído (seed fixa). Os números servem para comparar os métodos entre si, não como valor geral.
- A métrica não faz o pareamento um-para-um do BSDS (comentado no código).
- Tempo: Python puro nos laços do Watershed e de `unir_fragmentos`; imagens acima de ~1000 px de lado levam vários segundos, e convém reduzi-las.

**Por que não os outros métodos da aula:** o enunciado pede Watershed e SLIC. Os outros servem de contraste:
- **Limiarização (Otsu):** separa só por intensidade e gera regiões desconexas, sem noção de vizinhança; o carro azul, o fundo claro e os reflexos não se separam por um único limiar.
- **k-means puro:** agrupa só por cor; o SLIC é exatamente o k-means com a posição na distância e a busca local, o que gera regiões compactas e conexas.
- **Canny:** dá bordas, não regiões; as bordas não fecham sozinhas e não dá para contar regiões.
- **Crescimento de regiões:** depende de sementes manuais e de um limiar de similaridade; o Watershed com marcadores automáticos resolve as duas coisas pelo relevo do gradiente.